# ChemicalMotifIdentifier

This is a quick tutorial on:
1. Analytically obtaining a pattern inventory for a ternary system (CrCoNi) in the fcc crystal structure. 
2. Obtaining the pattern inventory with ML and creating a physically constrained embedding space from which we can compute dissimilarities between motifs. 
3. Chemical motif identification in atomistic data and computing dissimilarity between motifs.

In [22]:
%%capture 
! pip install gdown

! pip install chemicalmotifidentifier

! pip install rich

In [1]:
%%capture
from polya import Polya
from rich import print
from sympy import init_printing, symbols  # For latex formatting
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt

%matplotlib inline

from _frameworks import INPUT_GDOWN_LINK

# Downloading neceassry inputs (model weights, dump files, ...)
import gdown

# os.makedirs("data/", exist_ok=True)
# gdown.download_folder(INPUT_GDOWN_LINK, output="data/", quiet=True)

## 1. Ternary system pattern inventory

In [2]:
init_printing()

pl = Polya(graph_name="fcc")

ntypes = 3
p_g, nms = pl.get_gt(ntypes=ntypes)

# Replacing t1, t2 and t3 with Cr, Co, Ni.
p_g = p_g.subs(
    {
        symbols("t1"): symbols("Cr"),
        symbols("t2"): symbols("Co"),
        symbols("t3"): symbols("Ni"),
    }
)

print(
    "The pattern inventory for the fcc first coordination polyhedron of the CrCoNi system is given by: \n"
)

p_g

The pattern inventory for the fcc first coordination polyhedron of the CrCoNi system is given by:

In [3]:
print(
    f"\n We have a total of {nms} distinct coordination polyhedron, which in turns yield {nms*ntypes} distinct local chemical motifs. "
)

We have a total of 12111 distinct coordination polyhedron, which in turns yield 36333 distinct local chemical 
motifs.

## 2. Model expressivity on the fcc ternary synthetic dataset and physically constrained embedding space

In [4]:
%%capture 
from _frameworks import SyntheticChemicalMotifIdentifier

eca = SyntheticChemicalMotifIdentifier(crystal_structure="fcc")
df = eca.predict(
    root="data/synthetic/fcc_graph_datasets/",
    skeleton_graph_path="data/inputs_doi-10.48550-arXiv.2311.01545/fcc_1nn.pt",
    atom_types_paths=[
        "data/inputs_doi-10.48550-arXiv.2311.01545/fcc_nelement3_generators.pt"
    ],
    nelement=3,
)
os.makedirs("data/synthetic/outputs/", exist_ok=True)
df.to_pickle("data/synthetic/outputs/df_fcc.pkl")

In [4]:
# Obtaining the ML pattern inventory
df = pd.read_pickle("data/synthetic/outputs/df_fcc.pkl")

shell_concentrations, counts = (
    np.array(list(df.shell_concentration)),
    np.array(list(df.counts)),
)
unique_concentrations, counts = np.unique(
    shell_concentrations, axis=0, return_counts=True
)

pattern = {
    tuple(unique_concentrations[i]): counts[i]
    for i in range(len(unique_concentrations))
}
print("The machine learning pattern inventory is given by:")
print(pattern)
print(f"We have a total of {np.sum(counts)} distinct 1CP.")

The machine learning pattern inventory is given by:

{
    (0.0, 0.0, 12.0): 1,
    (0.0, 1.0, 11.0): 1,
    (0.0, 2.0, 10.0): 4,
    (0.0, 3.0, 9.0): 9,
    (0.0, 4.0, 8.0): 18,
    (0.0, 5.0, 7.0): 24,
    (0.0, 6.0, 6.0): 30,
    (0.0, 7.0, 5.0): 24,
    (0.0, 8.0, 4.0): 18,
    (0.0, 9.0, 3.0): 9,
    (0.0, 10.0, 2.0): 4,
    (0.0, 11.0, 1.0): 1,
    (0.0, 12.0, 0.0): 1,
    (1.0, 0.0, 11.0): 1,
    (1.0, 1.0, 10.0): 4,
    (1.0, 2.0, 9.0): 18,
    (1.0, 3.0, 8.0): 47,
    (1.0, 4.0, 7.0): 92,
    (1.0, 5.0, 6.0): 126,
    (1.0, 6.0, 5.0): 126,
    (1.0, 7.0, 4.0): 92,
    (1.0, 8.0, 3.0): 47,
    (1.0, 9.0, 2.0): 18,
    (1.0, 10.0, 1.0): 4,
    (1.0, 11.0, 0.0): 1,
    (2.0, 0.0, 10.0): 4,
    (2.0, 1.0, 9.0): 18,
    (2.0, 2.0, 8.0): 76,
    (2.0, 3.0, 7.0): 182,
    (2.0, 4.0, 6.0): 318,
    (2.0, 5.0, 5.0): 372,
    (2.0, 6.0, 4.0): 318,
    (2.0, 7.0, 3.0): 182,
    (2.0, 8.0, 2.0): 76,
    (2.0, 9.0, 1.0): 18,
    (2.0, 10.0, 0.0): 4,
    (3.0, 0.0, 9.0): 9,
    (3.0, 1.0, 8.0): 47,
    (3.0, 2.0, 7.0): 182,
    (3.0, 3.0, 6.0): 408,
    (3.0, 4.0, 5.0): 606,
    (3.0, 5.0, 4.0): 606,
    (3.0, 6.0, 3.0): 408,
    (3.0, 7.0, 2.0): 182,
    (3.0, 8.0, 1.0): 47,
    (3.0, 9.0, 0.0): 9,
    (4.0, 0.0, 8.0): 18,
    (4.0, 1.0, 7.0): 92,
    (4.0, 2.0, 6.0): 318,
    (4.0, 3.0, 5.0): 606,
    (4.0, 4.0, 4.0): 768,
    (4.0, 5.0, 3.0): 606,
    (4.0, 6.0, 2.0): 318,
    (4.0, 7.0, 1.0): 92,
    (4.0, 8.0, 0.0): 18,
    (5.0, 0.0, 7.0): 24,
    (5.0, 1.0, 6.0): 126,
    (5.0, 2.0, 5.0): 372,
    (5.0, 3.0, 4.0): 606,
    (5.0, 4.0, 3.0): 606,
    (5.0, 5.0, 2.0): 372,
    (5.0, 6.0, 1.0): 126,
    (5.0, 7.0, 0.0): 24,
    (6.0, 0.0, 6.0): 30,
    (6.0, 1.0, 5.0): 126,
    (6.0, 2.0, 4.0): 318,
    (6.0, 3.0, 3.0): 408,
    (6.0, 4.0, 2.0): 318,
    (6.0, 5.0, 1.0): 126,
    (6.0, 6.0, 0.0): 30,
    (7.0, 0.0, 5.0): 24,
    (7.0, 1.0, 4.0): 92,
    (7.0, 2.0, 3.0): 182,
    (7.0, 3.0, 2.0): 182,
    (7.0, 4.0, 1.0): 92,
    (7.0, 5.0, 0.0): 24,
    (8.0, 0.0, 4.0): 18,
    (8.0, 1.0, 3.0): 47,
    (8.0, 2.0, 2.0): 76,
    (8.0, 3.0, 1.0): 47,
    (8.0, 4.0, 0.0): 18,
    (9.0, 0.0, 3.0): 9,
    (9.0, 1.0, 2.0): 18,
    (9.0, 2.0, 1.0): 18,
    (9.0, 3.0, 0.0): 9,
    (10.0, 0.0, 2.0): 4,
    (10.0, 1.0, 1.0): 4,
    (10.0, 2.0, 0.0): 4,
    (11.0, 0.0, 1.0): 1,
    (11.0, 1.0, 0.0): 1,
    (12.0, 0.0, 0.0): 1
}

We have a total of 12111 distinct 1CP.

## 3. Chemical motif identification in atomistic data

In [5]:
%%capture 
from _frameworks import MonteCarloChemicalMotifIdentifier

dump_files = [
    f"data/inputs_doi-10.48550-arXiv.2311.01545/dumps/ordered_relaxation_20_{i}_300K.dump"
    for i in range(1, 5 + 1)
]

# need to replace the data/synthetic/outputs/df_fcc.pkl to data/inputs_doi-10.48550-arXiv.2311.01545/df_{self.crystal_structure}.pkl
eca = MonteCarloChemicalMotifIdentifier(crystal_structure="fcc")
df = eca.predict(root="data/mc/outputs/eca_id/300K/", dump_file=dump_files)
kl = eca.get_kl(df)
df.to_pickle("data/mc/outputs/eca_id/300K/df_microstates.pkl")

Note here that edges will be snap back to their closest perfect ideal direction. 

Alternatively, you can use the ``_frameworks.PTMMotifIdentifier`` to only identify motifs in specific crystal structures and for correcting their distortions.

Let's compute the dissimilarity between motif $\mathcal{M}_0$ and $\mathcal{M}_1$ shown below: 

In [11]:
from chemicalmotifidentifier import Plot
plot = Plot(
    structure="fcc",
    graph_folder="data/inputs_doi-10.48550-arXiv.2311.01545/graph_plot_templates",
)
plot.set_colors(np.array(["#FED700", "#21B0FE", "#FE218B"]))
plot.set_node_size(4)
plot.set_width(0.2)

for i in [0, 1]:
    df_row = df.iloc[i]

    types_with_central_atom = np.concatenate(
        ([df_row.central_atom_type], np.array(list(df_row.shell_atomic_types)))
    )

    fig, ax = plt.subplots(figsize=(1, 1))
    plot.plot_ms(new_atom_types=types_with_central_atom)
    plt.show()
    plt.savefig(f"data/inputs_doi-10.48550-arXiv.2311.01545/graph_plot_templates/{i}.png")

C:\Users\huangjiaxin\AppData\Local\Temp\ipykernel_25584\3903030449.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.


In [10]:
from chemicalmotifidentifier import Dissimilarity

phys_emb_i = np.concatenate(([df.iloc[0].central_atom_type], df.iloc[0].shell_phys_emb))
phys_emb_j = np.concatenate(([df.iloc[1].central_atom_type], df.iloc[1].shell_phys_emb))

(
    central_atom_dissim,
    concentration_dissim,
    structural_dissim,
) = Dissimilarity().get_separate_dissimilarities(3, phys_emb_j, phys_emb_i)

weights = np.array([12.0, 24.0, 24.0])
weights *= 1 / np.sum(weights)
d_ij = (
    weights[0] * central_atom_dissim
    + weights[1] * concentration_dissim / 12
    + weights[2] * structural_dissim
)


print(f"The dissimilarity between these two motifs is: {d_ij[0]:.2f}.\n\nBefore weighthing based on the number of bonds of each structure, they have a central atom dissimilarity of {central_atom_dissim[0]:.2f} since their central atom atomic types are the same. Their chemical composition dissimilarity is of {concentration_dissim[0]:.2f}/12 because we lost one red and gained one blue atom. And their structural dissimilarity is of {structural_dissim[0]:.2f}.")

The dissimilarity between these two motifs is: 0.05.

Before weighthing based on the number of bonds of each structure, they have a central atom dissimilarity of 0.00 
since their central atom atomic types are the same. Their chemical composition dissimilarity is of 1.00/12 because 
we lost one red and gained one blue atom. And their structural dissimilarity is of 0.04.

In [2]:
%%capture 
from _frameworks import MonteCarloChemicalMotifIdentifier

dump_files = [
    f"data/inputs_doi-10.48550-arXiv.2311.01545/dumps_own/ordered_relaxation_20_{i}_300K_shuffled.dump"
    for i in range(1, 2 + 1)
]

# need to replace the data/synthetic/outputs/df_fcc.pkl to data/inputs_doi-10.48550-arXiv.2311.01545/df_{self.crystal_structure}.pkl
eca = MonteCarloChemicalMotifIdentifier(crystal_structure="fcc")
df = eca.predict(root="data/mc/outputs/eca_id/300K/", dump_file=dump_files)
kl = eca.get_kl(df)
df.to_pickle("data/mc/outputs/eca_id/300K/df_microstates.pkl")

from chemicalmotifidentifier import Plot
plot = Plot(
    structure="fcc",
    graph_folder="data/inputs_doi-10.48550-arXiv.2311.01545/graph_plot_templates",
)
plot.set_colors(np.array(["#FED700", "#21B0FE", "#FE218B"]))
plot.set_node_size(4)
plot.set_width(0.2)

for i in [0, 1]:
    df_row = df.iloc[i]

    types_with_central_atom = np.concatenate(
        ([df_row.central_atom_type], np.array(list(df_row.shell_atomic_types)))
    )

    fig, ax = plt.subplots(figsize=(1, 1))
    plot.plot_ms(new_atom_types=types_with_central_atom)
    plt.show()
    plt.savefig(f"data/inputs_doi-10.48550-arXiv.2311.01545/graph_plot_templates/result_{i}.png")